# 从采集记录到知识候选：逐算子数据流

主线以**采集文件中的每条数据**为输入，不循环按概念检索。指定ID和采样只在第4步影响哪些记录进入后续处理；文档在第5、6步读取并清洗，之后才按概念建立关联并汇集。

```text
逐条读采集文件 → 从概念文件记录语言页面ID → 附加已有概念关联
→ [可选：ID过滤＋固定种子采样]
→ 逐条读正文／验图片字节 → 逐条清洗文档
→ 按概念建立材料关联 → 分窗口读取相关材料
→ 判定身份 → 去重和截取正文 → 提取知识 → 修订候选
→ 判断图片支持 → 保存候选／阻塞
```

**小批与全量共用同一套算子**：`IDS=None`关闭ID过滤，`SAMPLE_RATE=1.0`关闭采样，`MAX_RECORDS_PER_SOURCE=None`读取指定文件的全部记录。`SOURCE_PATHS=None`选择所有已支持的datasets来源。全量仍需为实际资源与模型调用设置预算，本次不启动全量或出题。

每阶段经demiflow执行，记录落入运行目录的SQLite表，后续通过游标读取，不将全表转成Python列表。查看时才limit100或固定种子抽样。原始采集行保存一次，后续阶段主要保存新增字段并通过record_id读取原始行。

**范围限制**：已有概念关联不是语义审核；没有关联或有歧义的材料保留。清洗仍是保守初版；按概念分出的窗口独立产出机器候选，尚未实现跨窗口冲突整合或最终知识库人工验收。不能把共用全量执行逻辑等同于全库质量已验收。


## 配置与读取方式

默认`MODE='view_saved'`只读本次已保存结果；要执行算子，设为`run`并使用新的RUN_NAME。`RUN_MODEL_CELLS=False`默认不发模型请求。修改显示cell不需要重新扫描；修改处理源码／配置需新run并重载代码。

| 输入／输出字段 | 含义 |
| --- | --- |
| `IDS` | 可选legacy:概念名或qid:QID列表；None允许所有材料。关联多概念的选中记录保留全部原始关联。 |
| `SAMPLE_RATE / SEED` | 按稳定record_id哈希抽取记录的比例与种子；与处理顺序无关。不是按概念分层随机抽样。 |
| `MAX_RECORDS_PER_SOURCE` | 可选每源读入上限；None表示读取完整文件。截断会记入source_status。 |
| `GROUP_SIZE` | 每个概念一次汇集的材料条数上限；多余材料进入后续窗口，不直接丢弃。 |
| `MODEL_CONFIG` | 现有知识算子的生成、片段和图片预算；max_cases不用于新的流式入口。 |
| `MODE` | view_saved读取旧manifest和表；run实例化真实执行流程。只读时当前配置不覆盖旧配置。 |


In [ ]:
from pathlib import Path
import sys, importlib, itertools, json, sqlite3
PROJECT = Path('/yzp/zhaozy/yangzepeng/0905/demiwtg')
if str(PROJECT) not in sys.path: sys.path.insert(0,str(PROJECT))
import curation.v4.record_flow as record_flow
import curation.v4.notebook_debug as notebook_debug
importlib.reload(record_flow)
importlib.reload(notebook_debug)
from curation.v4.record_flow import RecordFlow, RecordStore
from curation.v4.notebook_debug import show, detail
from curation.v4.pipeline import DEFAULT
MODE = 'view_saved'
RUN_NAME = 'record_stream_pilot_v1'
RUN = PROJECT/'state/curation/v4'/RUN_NAME
DATASET = PROJECT/'datasets/demiwtg'
IDS = ['legacy:木兰','legacy:芦笙','qid:Q1']  # 全部材料：None
SAMPLE_RATE = 1.0
SEED = 42
MAX_RECORDS_PER_SOURCE = 50_000  # 完整文件：None
GROUP_SIZE = 32
SOURCE_PATHS = [DATASET/x for x in ['meta/concepts.json','meta/qid_concepts.fat.jsonl.gz',
    'meta/docs.jsonl','meta/images.jsonl','corpus/pages-en-part1.jsonl.gz']]  # 所有支持来源：None
RUN_MODEL_CELLS = False
MODEL_CONFIG = {**DEFAULT,'max_calls':12}
assert MODE in {'view_saved','run'}
flow = (RecordFlow.open_saved(RUN) if MODE=='view_saved' else
        RecordFlow(RUN,DATASET,SOURCE_PATHS,IDS,SAMPLE_RATE,SEED,MAX_RECORDS_PER_SOURCE,GROUP_SIZE,PROJECT))
async def run_step(stage):
    if MODE=='view_saved':
        print(stage, '只读已有输出；不调用算子')
    else:
        print(await flow.step(stage))
async def run_knowledge(stage):
    if MODE=='run' and RUN_MODEL_CELLS:
        print(await flow.knowledge_step(stage,MODEL_CONFIG))
    else:
        print(stage,'只读已有输出；不调用模型')
def preview(stage, limit=100, sample=False, seed=42, selected_only=False):
    # 抽样先只读取ID，不扫描／反序列化所有正文；最后只取选中行的全文。
    import random
    if limit < 0: raise ValueError('limit must be nonnegative')
    if limit == 0: return []
    store=RecordStore(RUN)
    try:
        sql='SELECT id FROM outputs WHERE stage=?'
        if selected_only: sql += " AND json_extract(body, '$.entry_selection.selected')=1"
        sql += ' ORDER BY id'
        ids=[];rng=random.Random(seed)
        if not sample:
            ids=[r[0] for r in store.db.execute(sql+' LIMIT ?', (stage,limit))]
        else:
            for i,(key,) in enumerate(store.db.execute(sql,(stage,))):
                if len(ids)<limit: ids.append(key)
                else:
                    j=rng.randrange(i+1)
                    if j<limit: ids[j]=key
        return [store.get(stage,key) for key in ids]
    finally: store.close()
print('运行目录',RUN)
print('本运行实际入口配置：');detail(flow.config)


## 1．逐条读取采集文件

**输入**：`flow.sources`中的采集文件路径及类型；没有概念请求列表。JSONL／gzip按行读取，concepts.json使用流式JSON解析。

**动作**：解析每条数据、记录位置和哈希，保存原始内容。坏行单独保存到parse_errors；达到读入上限时记录未完整扫描。此步没有按概念排除记录。

**输出**：`read_records`表，每行是一条采集记录；另有`source_status`与`parse_errors`。

| 输入／输出字段 | 含义 |
| --- | --- |
| `record_id` | 该来源版本、位置及原始内容绑定的记录标识；同一条数据在所有阶段用同一ID。 |
| `kind` | legacy_concepts、qid_concepts、legacy_docs、legacy_images、wiki_pages等来源类型。 |
| `record` | 完整采集内容；文档清单里通常只有正文路径，Wiki记录本身包含sections。 |
| `provenance` | source_path文件路径；line或record_index从1开始；source_snapshot是文件大小、修改时间等；record_sha256为解析后内容哈希，逐行文件另存raw_line_sha256。 |


文件读取报告的字段：path文件路径；kind类型；read已读入记录数；valid成功解析数；invalid坏行数；complete是否读到末尾；finished是否完成本轮读取（可能因预算结束，不能代替complete）；status为read、budget_limited或missing。坏行表中的error说明解析失败，raw保留原始行。


In [ ]:
await run_step('read_records')

In [ ]:
raw_sample = preview('read_records',limit=100,sample=True,seed=42)
show([{'record_id':r['record_id'],'kind':r['kind'],'title':r['record'].get('title'),
       'name':r['record'].get('name'),'qid':r['record'].get('qid'),
       'provenance':r['provenance']} for r in raw_sample],limit=100)
if raw_sample: detail(raw_sample[0])
with sqlite3.connect(RUN/'records.sqlite') as db:
    source_reports=[json.loads(r[0]) for r in db.execute('SELECT body FROM source_status')]
show(source_reports,limit=100)
show(preview('parse_errors',limit=100))

## 2．从概念数据记录名称、QID和语言页面ID

**输入**：第1步中kind为概念信息的记录，包括本轮没有指定的概念。这里不按IDS或采样过滤，以免丢掉后续Wiki页面关联所需的映射。

**动作**：把概念名／QID与原始record_id关联；把QID文件的en／zh页面ID写入可查询的磁盘表。不是为每个请求重新扫描文件。

**输出**：`index_concepts`表、`concepts`关联表和`pages`映射表。页面对应多个QID时全部保留，不任意取一个。

| 输入／输出字段 | 含义 |
| --- | --- |
| `record_id` | 第1步中的概念信息记录ID。 |
| `concept_refs` | 已有名称／QID，形式为legacy:名称或qid:QID；不是已经审核合并的内部身份。 |
| `pages.lang / page_id / ref` | 语言、页面ID及其对应qid:QID；用于没有qid的Wiki页面。 |


In [ ]:
await run_step('index_concepts')

In [ ]:
show(preview('index_concepts',limit=100,sample=True,seed=42))

## 3．给每条文档和图片附加已有概念名／QID

**输入**：第1步中所有支持的材料记录，以及第2步的语言页面ID映射。不是请求列表。

**动作**：读取文档／图片原有concepts、instances或qid；缺qid的Wiki页面查语言＋page_id。此步只解释已有字段，不判断正文是否真的属于该概念。

**输出**：每条材料增加如下字段，原始内容仍可通过record_id读取。

| 输入／输出字段 | 含义 |
| --- | --- |
| `concept_refs` | 材料已有的全部概念关联；空列表不等于材料无价值。 |
| `association_method` | source_fields表示按原字段；language_page_id表示按页面映射。 |
| `association_status` | source_association_only=有来源关联；unassociated=没有关联；ambiguous_page_mapping=同语言页面对应多个QID。 |


In [ ]:
await run_step('attach_concept_ids')

In [ ]:
show(preview('attach_concept_ids',limit=100,sample=True,seed=42),columns=['record_id','kind','concept_refs','association_method','association_status'])

## 4．可选：按ID过滤并按固定种子采样材料

**输入**：第3步的逐条材料，以及IDS、SAMPLE_RATE、SEED。

**动作**：有IDS时保留至少命中一个ID的记录，再按记录哈希采样。IDS=None且采样比例1时所有材料通过，包括无概念关联的材料。选中后保留该记录的全部关联，不只保留命中的ID。

**输出**：所有选择决定写入select_input；后面只读取selected=true的数据。**这是唯一决定小批选哪些材料的算子**。第1步读入上限另作入口预算记录。

| 输入／输出字段 | 含义 |
| --- | --- |
| `entry_selection.selected` | true进入后续读正文与清洗；false保留筛选决定，不进入后续。 |
| `entry_selection.reason` | selected通过；id_filter未命中指定ID；hash_sample未被抽中。 |
| `concept_refs` | 仍为第3步的全部原始关联，筛选不修改。 |


In [ ]:
await run_step('select_input')

In [ ]:
show(preview('select_input',limit=100,sample=True,seed=42),columns=['record_id','kind','concept_refs','entry_selection'])
# 只看通过入口过滤的记录，不重新执行过滤：
selected_sample=preview('select_input',limit=100,selected_only=True)
show(selected_sample,columns=['record_id','kind','concept_refs','entry_selection'])

## 5．逐条读取文档正文、核对图片字节

**输入**：第4步selected=true的材料记录。此后算子均不接收IDS或采样参数。

**动作**：文档按record.path读取本地正文；Wiki保留已有sections；图片按path和sha256检查本地文件。缺文件或解码失败保存状态，不丢弃材料。关联多个概念不会触发多次读取。

**输出**：read_documents，每条输入仍对应一条输出。

| 输入／输出字段 | 含义 |
| --- | --- |
| `document.text / path / sha256` | 实际读取的全文、绝对文件路径和字节哈希。 |
| `document.status / error` | readable可读；read_error失败及原因。 |
| `bytes` | 图片文件检查对象：verified_bytes哈希与解码通过；not_local未找到；其他状态说明路径、哈希或解码错误。不是视觉身份判断。 |


In [ ]:
await run_step('read_documents')

In [ ]:
show(preview('read_documents',limit=100,sample=True,seed=42),columns=['record_id','kind','concept_refs','document','bytes'])

## 6．逐条清除文档外壳并记录原文位置

**输入**：第5步逐条材料。还没有按概念复制材料或汇集文章。

**动作**：复用现役CleanDocumentRecord／cleaning.py，清除明确导航、空行和布局模板；保留标题、表格、链接、图片线索和未知模板。每条采集文档只经过一次此算子，即使它关联多个概念。图片及其他非正文材料直接保留。

**输出**：clean_documents，文档增加cleaning，原始record和document仍可读取。

| 输入／输出字段 | 含义 |
| --- | --- |
| `cleaning.text` | 后续使用的清洗正文。 |
| `cleaning.status / warnings` | cleaned_candidate机械候选；needs_review有提示；unavailable没拿到正文。提示不会被当作事实错误。 |
| `cleaning.counts` | 输入／输出字符数、内容块数和排除块数（含空行）。 |
| `cleaning.blocks` | raw_start/raw_end/raw_text为源位置与原文；text为可读文字；decision与reason说明保留／排除；clean_start/clean_end为清洗正文位置；links/images保留线索。 |
| `cleaning.source_sha256 / source_locator / version` | 源文本哈希、来源定位及清洗版本；Wiki位置基于明确的章节串联方式。 |


In [ ]:
await run_step('clean_documents')

In [ ]:
clean_sample=preview('clean_documents',limit=100,sample=True,seed=42)
show([{'record_id':r['record_id'],'kind':r['kind'],'concept_refs':r['concept_refs'],
       'title':r['record'].get('title'),'cleaning':{k:v for k,v in r.get('cleaning',{}).items() if k in ['status','counts','warnings']}} for r in clean_sample])
text_sample=[r for r in clean_sample if 'cleaning' in r]
CLEAN_INDEX=0
if text_sample:
    chosen=text_sample[CLEAN_INDEX]
    from curation.v4.cleaning import raw_text
    detail({'title':chosen['record'].get('title'),'original':raw_text(chosen),
            'cleaned':chosen['cleaning']['text'],'warnings':chosen['cleaning']['warnings']})
    show(chosen['cleaning']['blocks'],limit=100,columns=['kind','raw_start','raw_end','raw_text','text','decision','reason'])

## 7．按已有概念关联保存文档和图片的record_id

**输入**：第6步清洗后的材料及其concept_refs。

**动作**：在磁盘associations表中保存概念—record_id关系，不重新读取或清洗材料。一条材料可建立多条关联；无关联／页面映射歧义者仍保留在输出表，但不送入普通概念提取。

**输出**：group_materials描述每条材料的关联状态；associations供后续按概念、按GROUP_SIZE窗口流式读取。某概念超过窗口大小的材料继续进入下一个窗口，不在入口被扔掉。

| 输入／输出字段 | 含义 |
| --- | --- |
| `record_id / concept_refs` | 已清洗材料及可用于汇集的已有概念关联。 |
| `status` | linked已建立关系；unassociated无概念关联；ambiguous_page_mapping映射有歧义。 |
| `record_window（汇集后）` | 某概念的材料窗口序号，从0开始；不同窗口尚未做语义冲突整合。 |
| `bundle / cleaned_materials（汇集后）` | 原始内容与清洗版本，交给现有知识算子；concept_id来自独立试运行注册表。 |


In [ ]:
await run_step('group_materials')

In [ ]:
show(preview('group_materials',limit=100,sample=True,seed=42))

## 从材料关系读取一个处理窗口（仅查看已有关联）

下面只显示概念关联数量，不需要把所有文章加载进内存。实际身份算子通过iter_groups按窗口读取；同一记录关联多个概念时复用此前清洗结果。构建窗口会使用内部ID注册表，因此默认只读查看不调用它。


In [ ]:
with sqlite3.connect(RUN/'records.sqlite') as db:
    group_counts=[{'concept_ref':r[0],'materials':r[1]} for r in db.execute(
        'SELECT ref,COUNT(*) FROM associations GROUP BY ref ORDER BY ref LIMIT 100')]
show(group_counts)


## 8．判定概念歧义，筛选可关联材料

输入：按已有概念汇集的一窗口原文与清洗材料，加相应概念信息。模型只收到最多12篇500字符清洗预览和最多2条图片元数据，不看像素。输出：identity状态、target_label、接受／拒绝及理由、实际预览材料和未查ID。当前关联筛选仍有二分状态和预览不足的限制。

| 输入／输出字段 | 含义 |
| --- | --- |
| `record_id / case_id` | 当前材料窗口的标识，绑定概念及记录ID列表。 |
| `bundle.request` | 窗口对应的概念名或QID；不是入口查询条件。 |
| `blocked` | 前一步阻塞信息；中间阶段跳过，export仍记录。 |
| `result` | 下方投影展示该步骤的结果对象；完整输出保存在同名SQLite阶段及stages目录。 |


本批尚未调用模型。默认只查看已有输出；真正执行需要MODE=run且RUN_MODEL_CELLS=True。

In [ ]:
await run_knowledge('identity')

In [ ]:
show([{'case_id':r['case_id'],'request':r['bundle']['request'],'blocked':r.get('blocked'),'result':r.get('identity')} for r in preview('identity',limit=100)],limit=100)

## 9．去重、按预算选文并截取正文

输入：身份已接受的材料。程序去重正文／图片、按页面与材料ID顺序选最多2篇，每篇最多6,500字符、合计13,000字符，并准备图片。输出material_pack：passages片段及引文位置、images可用图片、duplicates重复关系、omissions未读范围、image_gaps缺图。不是按来源权威性选文，后续仍需改进章节覆盖。

| 输入／输出字段 | 含义 |
| --- | --- |
| `record_id / case_id` | 当前材料窗口的标识，绑定概念及记录ID列表。 |
| `bundle.request` | 窗口对应的概念名或QID；不是入口查询条件。 |
| `blocked` | 前一步阻塞信息；中间阶段跳过，export仍记录。 |
| `result` | 下方投影展示该步骤的结果对象；完整输出保存在同名SQLite阶段及stages目录。 |


本批尚未调用模型。默认只查看已有输出；真正执行需要MODE=run且RUN_MODEL_CELLS=True。

In [ ]:
await run_knowledge('organize')

In [ ]:
show([{'case_id':r['case_id'],'request':r['bundle']['request'],'blocked':r.get('blocked'),'result':r.get('material_pack')} for r in preview('organize',limit=100)],limit=100)

## 10．从多篇正文提取带引文的知识

输入模型：concept及passages中的source_id、text、source_family、start、end；不传全部原文映射。输出extraction：facts（fact_id、statement、conditions、exceptions、evidence中的source_id与quote），unresolved_conflicts（source_ids、issue、needed_evidence）和coverage_note。引文匹配不等于正确。

| 输入／输出字段 | 含义 |
| --- | --- |
| `record_id / case_id` | 当前材料窗口的标识，绑定概念及记录ID列表。 |
| `bundle.request` | 窗口对应的概念名或QID；不是入口查询条件。 |
| `blocked` | 前一步阻塞信息；中间阶段跳过，export仍记录。 |
| `result` | 下方投影展示该步骤的结果对象；完整输出保存在同名SQLite阶段及stages目录。 |


本批尚未调用模型。默认只查看已有输出；真正执行需要MODE=run且RUN_MODEL_CELLS=True。

In [ ]:
await run_knowledge('extract')

In [ ]:
show([{'case_id':r['case_id'],'request':r['bundle']['request'],'blocked':r.get('blocked'),'result':r.get('extraction')} for r in preview('extract',limit=100)],limit=100)

## 11．对照原文修订候选并记录未解冲突

输入：同一批passages及上一轮extraction候选。同一本地模型复查，不自动补读全文。输出knowledge字段与extraction结构相同，增加changes（fact_id、action、reason）。它不是独立审核，也没有检查其他窗口的冲突。

| 输入／输出字段 | 含义 |
| --- | --- |
| `record_id / case_id` | 当前材料窗口的标识，绑定概念及记录ID列表。 |
| `bundle.request` | 窗口对应的概念名或QID；不是入口查询条件。 |
| `blocked` | 前一步阻塞信息；中间阶段跳过，export仍记录。 |
| `result` | 下方投影展示该步骤的结果对象；完整输出保存在同名SQLite阶段及stages目录。 |


本批尚未调用模型。默认只查看已有输出；真正执行需要MODE=run且RUN_MODEL_CELLS=True。

In [ ]:
await run_knowledge('consolidate')

In [ ]:
show([{'case_id':r['case_id'],'request':r['bundle']['request'],'blocked':r.get('blocked'),'result':r.get('knowledge')} for r in preview('consolidate',limit=100)],limit=100)

## 12．逐图逐知识判定支持范围

输入：knowledge.facts和material_pack.images的实际像素；原图核对后缩放到最长边1,024并JPEG编码。输出image_evidence：无图／无事实为not_run；实际调用后为machine_reviewed，内含images的caption和support逐图逐事实的status、region、supports、limitations。真实图像模型分支仍待验收。

| 输入／输出字段 | 含义 |
| --- | --- |
| `record_id / case_id` | 当前材料窗口的标识，绑定概念及记录ID列表。 |
| `bundle.request` | 窗口对应的概念名或QID；不是入口查询条件。 |
| `blocked` | 前一步阻塞信息；中间阶段跳过，export仍记录。 |
| `result` | 下方投影展示该步骤的结果对象；完整输出保存在同名SQLite阶段及stages目录。 |


本批尚未调用模型。默认只查看已有输出；真正执行需要MODE=run且RUN_MODEL_CELLS=True。

In [ ]:
await run_knowledge('evidence')

In [ ]:
show([{'case_id':r['case_id'],'request':r['bundle']['request'],'blocked':r.get('blocked'),'result':r.get('image_evidence')} for r in preview('evidence',limit=100)],limit=100)

## 13．保存知识候选、图片支持及阻塞

输入：身份判断、knowledge.facts及unresolved_conflicts、image_evidence和blocked。输出export及candidates文件：case_id、concept_id、request、status、facts、冲突与图片支持。blocked保存失败；machine_candidates_ready_for_review仍是机器候选，不是已核验知识库。

| 输入／输出字段 | 含义 |
| --- | --- |
| `record_id / case_id` | 当前材料窗口的标识，绑定概念及记录ID列表。 |
| `bundle.request` | 窗口对应的概念名或QID；不是入口查询条件。 |
| `blocked` | 前一步阻塞信息；中间阶段跳过，export仍记录。 |
| `result` | 下方投影展示该步骤的结果对象；完整输出保存在同名SQLite阶段及stages目录。 |


本批尚未调用模型。默认只查看已有输出；真正执行需要MODE=run且RUN_MODEL_CELLS=True。

In [ ]:
await run_knowledge('export')

In [ ]:
show([{'case_id':r['case_id'],'request':r['bundle']['request'],'blocked':r.get('blocked'),'result':r.get('export')} for r in preview('export',limit=100)],limit=100)

## 14．查看各步条数与真实运行记录

`records.sqlite`保存原始内容、选择决定、各步输出、概念页面对应与材料关联。`record_manifest.json`冻结来源、入口配置及代码版本。`knowledge_config.json`在真正运行后半段时冻结模型预算；calls保存完整请求响应。原始输入字段在后续表中按record_id引用，不将全量正文复制到每个预览列表。

输出表`stage`是步骤名，`rows`是已保存记录数。原始记录数量、通过过滤数量、文档数量、概念关系数量、知识条数含义不同，不能混在一起统计。


In [ ]:
show(flow.counts(),limit=100)
print('模型请求数：',len(list((RUN/'calls').glob('*.request.json'))))

## 附录：采集文件原始字段


| 字段 | 含义 |
| --- | --- |
| `name / aliases / carriers / taxonomy` | concepts.json中的名称、别名、载体类型、分类路径快照；并非新审核结果。 |
| `qid / en / zh` | QID及英文／中文页面信息；en和zh内的title为标题，page_id为对应语言站点的页面ID。 |
| `p18 / p373` | QID来源给出的P18图片地址和P373 Commons分类名称；未核验图片支持关系。 |
| `concepts / instances` | 采集清单中已有的关联概念名称。字段名不同来自两类清单，不意味着语义已核验。 |
| `url / title / path（文档）` | 抓取地址、标题、相对于数据集的本地正文路径。 |
| `page_sha` | 采集页面定位标识，旧页面按URL哈希寻址；不能当作正文内容哈希。 |
| `authority` | 采集端的来源路由／类型标签（如wiki、serp），不是本次可靠性评分。 |
| `query / fetched_at` | 采集时使用的查询词／源记录的抓取时间值；原样保留。 |
| `n_images / n_passages` | 采集端报告的图片数／片段数；不等于本轮实际获取或输入模型的数量。 |
| `sha256 / path / ext / mime / size_bytes（图片）` | 源记录的图片内容哈希、数据集内路径、扩展名、MIME类型和文件字节数；需与bytes实际核验区分。 |
| `width / height / orig_width / orig_height` | 来源记录的宽高及原始宽高字段；本轮实际解码尺寸看bytes.dimensions。 |
| `content_url / landing_url` | 图片内容地址／图片所在网页地址；不保证当前仍可访问。 |
| `author / license / source` | 采集端记录的作者、许可和来源；未在本轮独立核验。 |
| `caption` | 源记录已有描述，可能是历史模型生成；不自动当作原始图注或像素事实。 |
| `queries / query_langs` | 图片采集使用的查询词及语言。 |
| `identity / focus / quality / kb_match / richness` | 图片记录携带的历史判断或打分字段；本流程原样保留，未重新计算，也未将其当当前审核依据。具体取值尺度依该记录生成版本，不从名称猜测分数含义。 |
| `lang / page_id / revision_id` | Wiki语言、页面ID和页面修订版本；page_id必须结合语言使用。 |
| `sections` | Wiki章节内容；每项保留标题title和正文text等原有字段。 |
| `is_disambig / is_redirect / redirect_target` | Wiki来源标记的消歧义页、重定向页及重定向目标。 |
| `links / link_count / images / categories（Wiki）` | Wiki来源解析得到的链接、链接数、图片及分类信息；不等于已下载图片数或本项目taxonomy。 |
| `parser_version / text_sha256 / byte_len` | Wiki采集解析器版本、采集端报告的文本哈希和字节长度；原样保留，不能替代本次cleaning.source_sha256。 |

该表按本批实际出现的字段编写。未经本轮核验的历史字段只解释来源与用途，不把其数值当作当前质量结论。